In [10]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [11]:
model = ChatOllama(
    model="llama3.2",
    temperature=0.7
)

In [12]:
# defining state of the chatbot
class ChatbotState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]  # list of messages in the conversation]


In [13]:
# defining node
def chat_bot(state: ChatbotState) -> ChatbotState:
    messages = state["messages"]
    response = model.invoke(messages)
    
    return {
        "messages": [response]  # return the response as a list of messages
    }

In [14]:
# checkpointer -> to save the state of the chatbot
checkpointer = InMemorySaver()

# define the state graph
graph = StateGraph(ChatbotState)

# adding nodes to the graph
graph.add_node('chat_bot', chat_bot)

# adding edges to the graph
graph.add_edge(START, 'chat_bot')
graph.add_edge('chat_bot', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [16]:
thread_id = 1
while True:
    user_msg = input("User: ")
    print(f"User: {user_msg}")
    
    config = {"configurable": {"thread_id": thread_id}}
    if user_msg.strip().lower() in ["exit", "quit"]:
        break
    
    response = chatbot.invoke({"messages": [HumanMessage(content=user_msg)]}, config=config)
    
    print(f"AI: {response['messages'][-1].content}")

User: hi
AI: It's nice to meet you. Is there something I can help you with or would you like to chat?
User: how are u
AI: I'm just a language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to help with any questions or topics you'd like to discuss!

How about you? How's your day going so far?
User: exit
